# PyTorch за 30 минут

**ML PRO · подготовка, по желанию.** Напоминание синтаксиса, а не курс по нейросетям. Цель: в лекции 5.1 не спотыкаться о `.to(device)`, `loss.backward()` и `state_dict`.

Строим логистическую регрессию на PyTorch на том же датасете оттока и сравниваем с sklearn из baseline-ноутбука. Модель заведомо простая: смысл в механике, а не в качестве.

1. Тензоры
2. Autograd
3. nn.Module и цикл обучения
4. Оценка и сравнение с sklearn
5. Сохранение и загрузка весов
6. device: CPU и GPU

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
print(torch.__version__, "| CUDA доступна:", torch.cuda.is_available())

2.13.0+cpu | CUDA доступна: False


## 1. Тензоры

Тензор это numpy-массив, который умеет считать градиенты и жить на GPU. Почти все операции называются так же, как в numpy.

In [2]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(x.shape, x.dtype)
print(x @ x.T)                       # матричное умножение
print(x.mean(), x.sum(dim=0))        # dim вместо axis
a = np.arange(6, dtype=np.float32).reshape(2, 3)
t = torch.from_numpy(a)              # без копирования данных
print(t.numpy() is not None, t.float().dtype)

torch.Size([2, 2]) torch.float32
tensor([[ 5., 11.],
        [11., 25.]])
tensor(2.5000) tensor([4., 6.])
True torch.float32


Три ловушки: `dtype` по умолчанию `float32`, а не `float64` как в numpy; `dim` вместо `axis`; тензор с градиентом нельзя превратить в numpy без `.detach()`.

## 2. Autograd

`requires_grad=True` включает запись операций. `backward()` считает производные по цепочке и складывает их в `.grad`. На этом стоит всё обучение.

In [3]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w * 3 - 1) ** 2               # d/dw = 2 * (3w - 1) * 3 = 30 при w = 2
loss.backward()
print("градиент:", w.grad.item())

w.grad.zero_()                        # градиенты накапливаются: перед новым шагом обнуляем
with torch.no_grad():                 # внутри блока запись выключена: так делают шаг оптимизатора и инференс
    w -= 0.1 * 30
print("w после шага:", w.item())

градиент: 30.0
w после шага: -1.0


## 3. nn.Module и цикл обучения

Данные те же, что в baseline: числовые признаки масштабируем, категориальные кодируем one-hot через pandas. Внутри PyTorch нет ColumnTransformer, поэтому предобработка делается снаружи, и это уже причина, по которой в проде обычно связка sklearn-препроцессинг плюс модель.

In [4]:
df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
y = (df["Churn"] == "Yes").astype(np.float32).values
X = df.drop(columns=["customerID", "Churn"])

numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
X = pd.get_dummies(X, columns=[c for c in X.columns if c not in numeric], dtype=np.float32)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

mean, std = X_tr[numeric].mean(), X_tr[numeric].std()      # статистики только по train
X_tr[numeric] = (X_tr[numeric] - mean) / std
X_te[numeric] = (X_te[numeric] - mean) / std

Xt_tr = torch.tensor(X_tr.values, dtype=torch.float32)
Xt_te = torch.tensor(X_te.values, dtype=torch.float32)
yt_tr = torch.tensor(y_tr).unsqueeze(1)
yt_te = torch.tensor(y_te).unsqueeze(1)
print(Xt_tr.shape, Xt_te.shape)

torch.Size([5634, 45]) torch.Size([1409, 45])


In [5]:
class LogReg(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)      # веса и смещение

    def forward(self, x):
        return self.linear(x)                       # логиты; сигмоида внутри функции потерь

model = LogReg(Xt_tr.shape[1])
criterion = nn.BCEWithLogitsLoss()                  # численно устойчивее, чем sigmoid + BCELoss
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(300):
    model.train()
    optimizer.zero_grad()                           # 1. обнулить градиенты
    logits = model(Xt_tr)                           # 2. прямой проход
    loss = criterion(logits, yt_tr)                 # 3. потери
    loss.backward()                                 # 4. градиенты
    optimizer.step()                                # 5. шаг
    if epoch % 50 == 0 or epoch == 299:
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xt_te), yt_te).item()
        print(f"эпоха {epoch:3d}  train loss {loss.item():.4f}  test loss {val_loss:.4f}")

эпоха   0  train loss 0.7663  test loss 0.7215
эпоха  50  train loss 0.4270  test loss 0.4383
эпоха 100  train loss 0.4184  test loss 0.4313
эпоха 150  train loss 0.4166  test loss 0.4290
эпоха 200  train loss 0.4159  test loss 0.4277


эпоха 250  train loss 0.4155  test loss 0.4269
эпоха 299  train loss 0.4152  test loss 0.4263


Пять строк цикла (`zero_grad`, forward, loss, `backward`, `step`) одинаковы для логистической регрессии и для трансформера с миллиардом параметров. Меняется только то, что внутри `forward`.

Здесь весь датасет помещается в один батч. На больших данных цикл идёт по `DataLoader` мини-батчами, а `model.train()` и `model.eval()` переключают слои вроде dropout и batchnorm.

## 4. Оценка и сравнение с sklearn

In [6]:
model.eval()
with torch.no_grad():
    proba = torch.sigmoid(model(Xt_te)).squeeze(1).numpy()
auc_torch = roc_auc_score(y_te, proba)

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
auc_sk = roc_auc_score(y_te, lr.predict_proba(X_te)[:, 1])
print(f"ROC-AUC  torch {auc_torch:.4f}   sklearn {auc_sk:.4f}")

ROC-AUC  torch 0.8370   sklearn 0.8421


Одна и та же модель, два фреймворка, одинаковое качество. PyTorch не делает модель лучше, он делает её произвольной по архитектуре и переносимой на GPU.

## 5. Сохранение и загрузка весов

Сохраняют `state_dict`, то есть словарь тензоров, а не объект модели целиком. Для загрузки нужен код класса: это плата за гибкость и одновременно защита от чужих pickle.

In [7]:
from pathlib import Path
Path("artifacts").mkdir(exist_ok=True)
torch.save(model.state_dict(), "artifacts/logreg_torch.pt")

restored = LogReg(Xt_tr.shape[1])
restored.load_state_dict(torch.load("artifacts/logreg_torch.pt", weights_only=True))
restored.eval()
with torch.no_grad():
    same = torch.allclose(restored(Xt_te), model(Xt_te))
print("восстановленная модель даёт те же выходы:", same)
print(list(model.state_dict().keys()))

восстановленная модель даёт те же выходы: True
['linear.weight', 'linear.bias']


Вместе с весами в проде сохраняют и параметры предобработки: здесь это `mean` и `std` числовых колонок и список колонок после `get_dummies`. Без них загруженная модель получит признаки в другом масштабе и порядке и будет молча отвечать мусором. Это тот же train-serve skew, что и в baseline-ноутбуке, только острее: в PyTorch предобработку никто за вас не упакует.

## 6. device: CPU и GPU

Код один и тот же, меняется одна строка. Тензоры и модель должны быть на одном устройстве, иначе ошибка `Expected all tensors to be on the same device`.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
batch = Xt_te[:5].to(device)
with torch.no_grad():
    out = torch.sigmoid(model(batch)).cpu().numpy()   # обратно на CPU перед numpy
print(device, out.ravel().round(3))

cpu [0.03  0.7   0.065 0.398 0.026]


## Что запомнить

- Тензор это numpy с градиентами и устройством; `dim`, а не `axis`; `float32` по умолчанию
- `zero_grad`, forward, loss, `backward`, `step`: пять строк любого обучения
- `model.eval()` и `torch.no_grad()` на инференсе, `model.train()` при обучении
- Сохраняют `state_dict` плюс параметры предобработки, загружают в тот же класс
- Модель и данные на одном `device`; `.cpu()` перед `.numpy()`

В неделе 5 к этому добавятся токенизатор, предобученные веса из HuggingFace и LoRA. Механика цикла останется той же.